
# Regression Analysis for Empirical Privacy Analysis

1. Builds both **app-level** and **context-level** regression tables.
2. Estimates:
   - **App-level OLS models** for the four privacy metrics.
   - **Context-level clustered OLS models** for metrics that vary across app--country observations.
   - **High-risk binomial models** using top-quartile thresholds.
   - **Panel-style sensitivity models** using **GEE** and **fixed-effects-style OLS**.
3. Produces compact result tables and coefficient plots.
4. Adds supporting analyses linking privacy metrics to monetization, popularity, permissions, and trackers.

In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("../data/mhealth_apps_metrics.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Could not find mhealth_apps_metrics.csv in ../data")

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and validate the dataset

In [ ]:

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Unique apps:", df["app_id"].nunique())
print("Countries:", df["country"].nunique() if "country" in df.columns else "N/A")
print("Categories:", df["categories"].nunique() if "categories" in df.columns else "N/A")

required_cols = [
    "app_id", "ADII", "DGI", "PCLR", "AS",
    "downloads_int", "ratings_count", "average_score",
    "num_permissions", "num_dangerous_permissions", "num_trackers",
    "offersIAP", "ad_supported", "top_grossing",
    "categories", "region", "country", "country_label", "price"
]
missing_required = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing_required if missing_required else "None")


## 2. Prepare analysis datasets

We build two related tables:

- **App-level table:** one row per app, used for the main OLS and high-risk models.
- **Context-level table:** one row per app--country observation, used for repeated-observation robustness checks.

In [ ]:

analysis_df = df.copy()

def to_bin(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(int)
    s = series.astype(str).str.strip().str.lower()
    mapped = s.map({
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "1": 1, "0": 0
    })
    return pd.to_numeric(mapped, errors="coerce")

# Normalize indicator variables
analysis_df["top_grossing_bin"] = to_bin(analysis_df["top_grossing"])
analysis_df["offersIAP_bin"] = to_bin(analysis_df["offersIAP"])
analysis_df["ad_supported_bin"] = to_bin(analysis_df["ad_supported"])

# Ensure numeric columns
numeric_cols = [
    "ADII", "DGI", "PCLR", "AS",
    "downloads_int", "ratings_count", "average_score",
    "num_permissions", "num_dangerous_permissions", "num_trackers", "price"
]
for col in numeric_cols:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

analysis_df["downloads_for_model"] = analysis_df["downloads_int"]

def mode_first(series):
    mode = series.mode(dropna=True)
    if not mode.empty:
        return mode.iloc[0]
    non_null = series.dropna()
    return non_null.iloc[0] if not non_null.empty else np.nan

# App-level aggregation
app_level = (
    analysis_df
    .groupby("app_id", as_index=False)
    .agg({
        "ADII": "mean",
        "DGI": "mean",
        "PCLR": "mean",
        "AS": "mean",
        "downloads_for_model": "mean",
        "ratings_count": "mean",
        "average_score": "mean",
        "num_permissions": "mean",
        "num_dangerous_permissions": "mean",
        "num_trackers": "mean",
        "offersIAP_bin": "max",
        "ad_supported_bin": "max",
        "top_grossing_bin": "max",
        "categories": mode_first,
        "region": mode_first,
        "country_label": "nunique",
    })
    .rename(columns={"country_label": "country_count"})
)

# Context-level table
context_level = analysis_df[[
    "app_id", "country", "country_label", "region", "categories",
    "ADII", "DGI", "PCLR", "AS",
    "downloads_for_model", "ratings_count", "average_score",
    "num_permissions", "num_dangerous_permissions", "num_trackers",
    "offersIAP_bin", "ad_supported_bin", "top_grossing_bin"
]].copy()

def add_log1p(df_in, cols):
    df_out = df_in.copy()
    for c in cols:
        if c in df_out.columns:
            df_out[f"log_{c}"] = np.log1p(pd.to_numeric(df_out[c], errors="coerce").clip(lower=0))
    return df_out

log_cols = ["ADII", "downloads_for_model", "ratings_count", "num_permissions", "num_dangerous_permissions", "num_trackers"]
app_level = add_log1p(app_level, log_cols)
context_level = add_log1p(context_level, log_cols)

print("App-level shape    :", app_level.shape)
print("Context-level shape:", context_level.shape)


## 3. Outcome coverage and distributions

Before fitting regression models, we inspect missingness and distributional shape.  
This matters because:
- **ADII** is strongly right-skewed, so the app-level continuous model uses `log_ADII`.
- **DGI**, **PCLR**, and **AS** already lie on bounded scales and are modeled directly in the main OLS specifications.


In [ ]:

summary = pd.DataFrame({
    "non_null": app_level[["ADII", "DGI", "PCLR", "AS"]].notna().sum(),
    "mean": app_level[["ADII", "DGI", "PCLR", "AS"]].mean(),
    "median": app_level[["ADII", "DGI", "PCLR", "AS"]].median(),
    "std": app_level[["ADII", "DGI", "PCLR", "AS"]].std(),
    "min": app_level[["ADII", "DGI", "PCLR", "AS"]].min(),
    "max": app_level[["ADII", "DGI", "PCLR", "AS"]].max(),
}).round(4)

display(summary)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6, 4))
axes = axes.ravel()

# Brighter, high-contrast colors
colors = ["#0072B2", "#E69F00", "#009E73", "#D55E00"]  
# blue, orange, teal, reddish-orange (colorblind-friendly)

for ax, col, color in zip(axes, ["ADII", "DGI", "PCLR", "AS"], colors):
    ax.hist(app_level[col].dropna(), bins=40, color=color, edgecolor="black", alpha=0.9)
    ax.set_title(col)

plt.tight_layout()
plt.savefig(FIG_DIR / "regression_outcome_distributions.png", bbox_inches="tight")
plt.show()


## 4. Modeling helpers

The paper's regression framework uses:
- **app-level OLS** for the main continuous analyses,
- **context-level clustered OLS** for repeated app--country observations,
- **binomial models** for threshold-style "high-risk" outcomes,
- and **panel-style sensitivity checks** to account for repeated observations of the same app.

In [ ]:

def standardize_columns(df_in, cols):
    df_out = df_in.copy()
    for c in cols:
        s = pd.to_numeric(df_out[c], errors="coerce")
        sd = s.std()
        if pd.notna(sd) and sd > 0:
            df_out[f"z_{c}"] = (s - s.mean()) / sd
        else:
            df_out[f"z_{c}"] = 0.0
    return df_out

standardize_base = [
    "log_downloads_for_model",
    "log_ratings_count",
    "average_score",
    "log_num_permissions",
    "log_num_dangerous_permissions",
    "log_num_trackers",
]

app_model_df = standardize_columns(app_level, standardize_base)
context_model_df = standardize_columns(context_level, standardize_base)

def tidy_ols_result(model, model_name, outcome):
    out = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t_or_z": model.tvalues.values if hasattr(model, "tvalues") else model.zvalues,
        "p_value": model.pvalues.values,
    })
    ci = model.conf_int()
    out["ci_low"] = ci[0].values
    out["ci_high"] = ci[1].values
    out["model"] = model_name
    out["outcome"] = outcome
    out["nobs"] = int(model.nobs)
    out["r_squared"] = getattr(model, "rsquared", np.nan)
    out["pseudo_r2"] = getattr(model, "prsquared", np.nan)
    return out

def tidy_binomial_result(model, model_name, outcome):
    params = model.params
    conf = model.conf_int()
    out = pd.DataFrame({
        "term": params.index,
        "coef_logodds": params.values,
        "odds_ratio": np.exp(params.values),
        "p_value": model.pvalues.values,
        "or_ci_low": np.exp(conf[0].values),
        "or_ci_high": np.exp(conf[1].values),
        "model": model_name,
        "outcome": outcome,
        "nobs": int(model.nobs),
        "pseudo_r2": getattr(model, "prsquared", np.nan)
    })
    return out

def significance_stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


## 4b. Multicollinearity check and reference-category note

Two things worth checking before interpreting the coefficients below:

- **Multicollinearity**: `log_downloads_for_model` and `log_ratings_count` are highly correlated (apps with more installs tend to have proportionally more ratings), which can inflate coefficient standard errors if both are included. Variance Inflation Factors (VIF) for all continuous predictors are reported below; a VIF above ~5-10 is conventionally treated as a concern.
- **Reference category**: `C(categories)` and `C(region)` use patsy's default treatment coding, whose reference (omitted) level is whichever category/region sorts first alphabetically -- currently "Education" for `categories` and "Africa" for `region`. Every category/region coefficient in the tables below should be read as "relative to that omitted level," not as an absolute effect.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = [
    "z_log_downloads_for_model",
    "z_log_ratings_count",
    "z_average_score",
    "z_log_num_permissions",
    "z_log_num_dangerous_permissions",
    "z_log_num_trackers",
    "offersIAP_bin",
    "ad_supported_bin",
    "top_grossing_bin",
]
vif_input = app_model_df[vif_cols].replace([np.inf, -np.inf], np.nan).dropna()

vif_table = pd.DataFrame({
    "predictor": vif_cols,
    "VIF": [
        variance_inflation_factor(vif_input.values, i)
        for i in range(len(vif_cols))
    ],
}).sort_values("VIF", ascending=False).reset_index(drop=True)

display(vif_table.round(2))

high_vif = vif_table[vif_table["VIF"] >= 5]
if not high_vif.empty:
    print(
        "Predictors with VIF >= 5 (potential multicollinearity concern):\n"
        + ", ".join(high_vif["predictor"])
    )
else:
    print("No predictor has VIF >= 5.")

print(
    "\nReference levels for factor variables (patsy default = "
    "alphabetically first level):"
)
if "categories" in app_model_df.columns:
    print(f"  categories: {sorted(app_model_df['categories'].dropna().unique())[0]!r}")
if "region" in context_model_df.columns:
    print(f"  region:     {sorted(context_model_df['region'].dropna().unique())[0]!r}")


## 5. Main app-level OLS models

These are the main continuous models for the paper.

- Outcome `log_ADII` is log-transformed because ADII is strongly right-skewed.
- Outcomes `DGI`, `PCLR`, and `AS` are modeled on their original scales.
- Category indicators are included as functional-domain controls.
- HC3 robust standard errors are used to reduce sensitivity to heteroskedasticity.


In [ ]:

ols_specs = {
    "log_ADII": "log_ADII ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
                "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
                "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)",
    "DGI": "DGI ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
           "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
           "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)",
    "PCLR": "PCLR ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
            "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
            "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)",
    "AS": "AS ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
          "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
          "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)"
}

model_predictors = [
    "z_log_downloads_for_model", "z_log_ratings_count", "z_average_score",
    "z_log_num_permissions", "z_log_num_dangerous_permissions", "z_log_num_trackers",
    "offersIAP_bin", "ad_supported_bin", "top_grossing_bin", "categories",
]

ols_models = {}
ols_tidy_list = []

for outcome, formula in ols_specs.items():
    model_df = app_model_df.copy().replace([np.inf, -np.inf], np.nan)
    model_df = model_df.dropna(subset=[outcome])

    n_before = len(model_df)
    missing_by_col = model_df[model_predictors].isna().sum()
    missing_by_col = missing_by_col[missing_by_col > 0]
    if not missing_by_col.empty:
        print(
            f"[{outcome}] {n_before} apps have a non-null outcome; "
            f"missing predictor values (rows may overlap): "
            + ", ".join(f"{c}={n}" for c, n in missing_by_col.items())
        )

    mod = smf.ols(formula=formula, data=model_df).fit(cov_type="HC3")
    print(f"[{outcome}] nobs used in fit: {int(mod.nobs)}/{n_before} apps with non-null outcome")
    ols_models[outcome] = mod
    ols_tidy_list.append(tidy_ols_result(mod, "App-level OLS (HC3)", outcome))
    print(f"===== {outcome} =====")
    display(pd.DataFrame({
        "coef": mod.params,
        "std_err": mod.bse,
        "p_value": mod.pvalues
    }).round(4).head(20))

ols_results = pd.concat(ols_tidy_list, ignore_index=True)

model_fit = pd.DataFrame({
    "outcome": list(ols_models.keys()),
    "nobs": [int(m.nobs) for m in ols_models.values()],
    "r_squared": [round(m.rsquared, 4) for m in ols_models.values()],
    "adj_r_squared": [round(m.rsquared_adj, 4) for m in ols_models.values()],
})
display(model_fit)



## 6. Context-level robustness models

The metrics `ADII`, `DGI`, and `PCLR` vary across app--country observations.  
To preserve that repeated-observation structure, we fit context-level OLS models and cluster the standard errors by `app_id`.

This provides a direct robustness check for the main app-level results.


In [ ]:

context_specs = {
    "log_ADII": "log_ADII ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
                "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
                "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories) + C(region)",
    "DGI": "DGI ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
           "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
           "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories) + C(region)",
    "PCLR": "PCLR ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
            "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
            "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories) + C(region)"
}

context_models = {}
context_tidy_list = []

for outcome, formula in context_specs.items():
    model_df = context_model_df.copy().replace([np.inf, -np.inf], np.nan)

    base_fit = smf.ols(formula=formula, data=model_df, missing="drop").fit()
    used_rows = base_fit.model.data.row_labels
    cluster_groups = model_df.loc[used_rows, "app_id"]

    mod = smf.ols(formula=formula, data=model_df, missing="drop").fit(
        cov_type="cluster",
        cov_kwds={"groups": cluster_groups}
    )
    context_models[outcome] = mod
    context_tidy_list.append(tidy_ols_result(mod, "Context-level OLS (clustered by app)", outcome))
    print(f"===== Context model: {outcome} =====")
    display(pd.DataFrame({
        "coef": mod.params,
        "std_err": mod.bse,
        "p_value": mod.pvalues
    }).round(4).head(20))

context_results = pd.concat(context_tidy_list, ignore_index=True)

context_fit = pd.DataFrame({
    "outcome": list(context_models.keys()),
    "nobs": [int(m.nobs) for m in context_models.values()],
    "r_squared": [round(m.rsquared, 4) for m in context_models.values()],
    "adj_r_squared": [round(m.rsquared_adj, 4) for m in context_models.values()],
})
display(context_fit)



## 7. High-risk binomial models

To support threshold-style interpretation in the paper, we define **high-risk** outcomes using the top quartile of each metric.

This merged notebook uses **binomial GLMs** instead of plain `logit` because the earlier draft could fail with singular Hessians under richer category controls. The GLM specification is more stable on this dataset while still yielding interpretable **odds ratios**.


In [ ]:

logit_df = app_model_df.copy()

for col in ["ADII", "PCLR", "AS"]:
    threshold = logit_df[col].quantile(0.75)
    logit_df[f"high_{col}"] = (logit_df[col] >= threshold).astype(int)
    n_flagged = int(logit_df[f"high_{col}"].sum())
    print(
        f"{col}: 75th percentile threshold = {threshold:.4f} -> "
        f"{n_flagged} apps ({n_flagged / len(logit_df) * 100:.1f}%) flagged"
    )

# DGI has a large point mass at its maximum: a substantial share of apps
# have DGI == 1.0 exactly (fully undisclosed -- none of the personal data
# actually observed in traffic is mentioned in the policy). That mass
# alone exceeds 25% of the sample, so the 75th percentile of DGI equals
# its maximum, and a ">=" threshold silently flagged the entire ceiling
# group as "high_DGI" -- roughly double the intended top-quartile size,
# and a much less extreme comparison group than "high_ADII"/"high_PCLR"/
# "high_AS" above. No quantile-based split can produce a genuine top
# quartile here, so "high_DGI" is instead defined directly as the ceiling
# group (DGI == max), which is itself a meaningful, well-defined high-risk
# category -- and its true size is reported rather than mislabeled as 25%.
dgi_max = logit_df["DGI"].max()
dgi_p75 = logit_df["DGI"].quantile(0.75)
logit_df["high_DGI"] = logit_df["DGI"].eq(dgi_max).astype(int)
n_flagged_dgi = int(logit_df["high_DGI"].sum())
print(
    f"DGI: 75th percentile ({dgi_p75:.4f}) equals the maximum ({dgi_max:.4f}); "
    f"a quantile-based top-quartile split is not achievable for this metric. "
    f"high_DGI is instead defined as DGI == max, flagging "
    f"{n_flagged_dgi} apps ({n_flagged_dgi / len(logit_df) * 100:.1f}%) -- "
    f"NOT a top-quartile group, unlike the other three outcomes above."
)

# AS is undefined (NaN) for apps observed in too few countries for a
# reliable pairwise-distance estimate; sibling notebook
# 07_within_app_adaptation_across_contexts.ipynb requires >= 3 countries
# before using AS for exactly this reason. That filter wasn't applied
# here, so "high_AS" previously mixed AS estimates of very different
# reliability (an app with only 2 country observations contributes a
# single pairwise comparison, versus up to 325 pairs for a 26-country
# app) with equal weight. Apply the same minimum-coverage filter here for
# consistency with 07.
MIN_COUNTRIES_FOR_AS = 3
if "country_count" in logit_df.columns:
    low_coverage_as = logit_df["country_count"] < MIN_COUNTRIES_FOR_AS
    n_dropped = int((low_coverage_as & logit_df["high_AS"].notna()).sum())
    logit_df.loc[low_coverage_as, "high_AS"] = np.nan
    print(
        f"high_AS: excluded {n_dropped} apps with < {MIN_COUNTRIES_FOR_AS} observed "
        f"countries (AS unreliable for these), matching notebook 07's MIN_COUNTRIES filter."
    )

binomial_specs = {
    "high_ADII": "high_ADII ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
                 "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
                 "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)",
    "high_DGI": "high_DGI ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
                "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
                "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)",
    "high_PCLR": "high_PCLR ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
                 "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
                 "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)",
    "high_AS": "high_AS ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
               "z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
               "offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories)"
}

binomial_models = {}
binomial_tidy_list = []

for outcome, formula in binomial_specs.items():
    model_df = logit_df.copy().replace([np.inf, -np.inf], np.nan)
    model_df = model_df.dropna(subset=[outcome])
    mod = smf.glm(formula=formula, data=model_df, family=sm.families.Binomial()).fit()
    binomial_models[outcome] = mod
    binomial_tidy_list.append(tidy_binomial_result(mod, "App-level Binomial GLM", outcome))
    print(f"===== {outcome} =====")
    display(pd.DataFrame({
        "coef_logodds": mod.params,
        "std_err": mod.bse,
        "p_value": mod.pvalues,
        "odds_ratio": np.exp(mod.params)
    }).round(4).head(20))

binomial_results = pd.concat(binomial_tidy_list, ignore_index=True)
binomial_fit = pd.DataFrame({
    "outcome": list(binomial_models.keys()),
    "nobs": [int(m.nobs) for m in binomial_models.values()],
    "deviance": [round(m.deviance, 4) for m in binomial_models.values()],
})
display(binomial_fit)



## 8. Compact paper-ready summaries

The next cells extract the main non-category predictors from the fitted models.  
These compact tables are useful for LaTeX, narrative writing, and cross-model comparison.


In [ ]:

focus_terms = [
    "z_log_downloads_for_model",
    "z_log_ratings_count",
    "z_average_score",
    "z_log_num_permissions",
    "z_log_num_dangerous_permissions",
    "z_log_num_trackers",
    "offersIAP_bin",
    "ad_supported_bin",
    "top_grossing_bin",
]

ols_focus = (
    ols_results[ols_results["term"].isin(focus_terms)]
    .copy()
    .pivot(index="term", columns="outcome", values="coef")
    .round(3)
)

ols_pvals = (
    ols_results[ols_results["term"].isin(focus_terms)]
    .copy()
    .pivot(index="term", columns="outcome", values="p_value")
    .round(4)
)

display(ols_focus)
display(ols_pvals)


In [ ]:

binomial_focus = (
    binomial_results[binomial_results["term"].isin(focus_terms)]
    .copy()
    .pivot(index="term", columns="outcome", values="odds_ratio")
    .round(3)
)

binomial_pvals = (
    binomial_results[binomial_results["term"].isin(focus_terms)]
    .copy()
    .pivot(index="term", columns="outcome", values="p_value")
    .round(4)
)

display(binomial_focus)
display(binomial_pvals)



## 9. Coefficient visualization

This figure summarizes the **app-level OLS coefficients** for the main non-category predictors across the four continuous outcomes.


In [ ]:
plot_df = ols_results[ols_results["term"].isin(focus_terms)].copy()

term_labels = {
    "z_log_downloads_for_model": "Log installs",
    "z_log_ratings_count": "Log ratings",
    "z_average_score": "Average score",
    "z_log_num_permissions": "Log permissions",
    "z_log_num_dangerous_permissions": "Log dangerous permissions",
    "z_log_num_trackers": "Log trackers",
    "offersIAP_bin": "Offers IAP",
    "ad_supported_bin": "Ad supported",
    "top_grossing_bin": "Top grossing",
}
plot_df["term_label"] = plot_df["term"].map(term_labels)

# Global consistent order
term_order = (
    plot_df.groupby("term_label")["coef"]
    .mean()
    .sort_values()
    .index.tolist()
)

y = np.arange(len(term_order))  # GLOBAL y

outcome_colors = {
    "log_ADII": "#1f77b4",
    "DGI": "#ff7f0e",
    "PCLR": "#2ca02c",
    "AS": "#d62728",
}

outcome_order = ["log_ADII", "DGI", "PCLR", "AS"]

fig, axes = plt.subplots(1, 4, figsize=(11, 3), sharey=True)

for i, (ax, outcome) in enumerate(zip(axes, outcome_order)):
    sub = plot_df[plot_df["outcome"] == outcome].copy()

    # Align to same order
    sub = sub.set_index("term_label").loc[term_order].reset_index()

    color = outcome_colors[outcome]

    ax.errorbar(
        sub["coef"], y,
        xerr=[sub["coef"] - sub["ci_low"], sub["ci_high"] - sub["coef"]],
        fmt="o",
        color=color,
        ecolor=color,
        elinewidth=2,
        capsize=3,
        markersize=6
    )

    ax.axvline(0, linestyle="--", linewidth=1.2, color="black", alpha=0.7)

    # --- FORCE ticks on all axes ---
    ax.set_yticks(y)

    if i == 0:
        ax.set_yticklabels(term_order)
    else:
        # keep ticks but hide labels (THIS avoids disappearing ticks)
        ax.tick_params(axis="y", labelleft=False)

    ax.set_ylim(-0.5, len(term_order) - 0.5)  # enforce same limits

    ax.set_title(outcome, fontsize=13, fontweight="bold")
    ax.set_xlabel("Coefficient (95% CI)")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / "app_level_ols_coefficients.png", bbox_inches="tight", dpi=600)
plt.show()


## 10. Panel-style sensitivity analysis

This section merges the stronger repeated-observation ideas from the second notebook.

Because each app is observed across multiple countries, we add two robustness checks:

1. **GEE with exchangeable correlation grouped by app**  
   This captures within-app dependence across repeated country observations without requiring the stronger distributional assumptions of a mixed-effects model.

2. **Fixed-effects-style OLS with app and country indicators**  
   This absorbs stable app-specific differences and isolates country-level contrasts.

These are treated as **sensitivity analyses**, not replacements for the main models above.


In [ ]:

gee_results = []
gee_models = {}

# Use log_ADII, not raw ADII: the main models above use log_ADII because
# ADII is strongly right-skewed (cell 3's markdown notes this explicitly).
# This GEE check previously regressed raw ADII, so it wasn't actually a
# robustness check of the same specification -- a Gaussian-family model on
# heavily right-skewed raw ADII (app-level max is roughly 20x the mean)
# is a materially different, less appropriate specification, and its
# coefficients aren't on a comparable scale to the main model's.
gee_targets = ["log_ADII", "DGI", "PCLR"]
for target in gee_targets:
    formula = (
        f"{target} ~ z_log_downloads_for_model + z_log_ratings_count + z_average_score + "
        f"z_log_num_permissions + z_log_num_dangerous_permissions + z_log_num_trackers + "
        f"offersIAP_bin + ad_supported_bin + top_grossing_bin + C(categories) + C(region)"
    )
    gee_input = context_model_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[target, "app_id"])
    model = smf.gee(
        formula=formula,
        groups="app_id",
        data=gee_input,
        family=sm.families.Gaussian(),
        cov_struct=sm.cov_struct.Exchangeable(),
    ).fit()
    gee_models[target] = model
    gee_results.append(tidy_ols_result(model, "GEE (grouped by app)", target))
    print(f"===== GEE model: {target} =====")
    display(pd.DataFrame({
        "coef": model.params,
        "std_err": model.bse,
        "p_value": model.pvalues
    }).round(4).head(20))

gee_results = pd.concat(gee_results, ignore_index=True)
gee_results.head()


In [ ]:

fe_summaries = {}
# Use log_ADII, not raw ADII, matching the GEE check above and the main
# models -- see the note in the GEE cell for why regressing raw ADII here
# would not actually be a robustness check of the same specification.
fe_targets = [m for m in ["log_ADII", "DGI", "PCLR"] if m in context_model_df.columns]

for target in fe_targets:
    formula = f"{target} ~ C(app_id) + C(country_label)"
    fe_input = context_model_df.dropna(subset=[target]).copy()

    try:
        fe_model = smf.ols(formula, data=fe_input).fit(cov_type="HC3")
        core_terms = [t for t in fe_model.params.index if not t.startswith("C(app_id)")]
        fe_table = pd.DataFrame({
            "term": core_terms,
            "coef": [fe_model.params[t] for t in core_terms],
            "p_value": [fe_model.pvalues[t] for t in core_terms],
            "target": target,
            "n_obs": int(fe_model.nobs),
            "r_squared": float(fe_model.rsquared),
        })
        fe_summaries[target] = fe_table
        print(f"===== Fixed-effects-style model for {target} =====")
        display(fe_table.head(20))
    except Exception as e:
        print(f"Could not estimate fixed-effects model for {target}: {e}")

if fe_summaries:
    fe_results = pd.concat(fe_summaries.values(), ignore_index=True)
else:
    fe_results = pd.DataFrame()



## 11. Monetization, popularity, permissions, and trackers

This section keeps the useful business-model and ecosystem analyses from the second notebook, but integrates them into the same cleaned workflow.

These summaries are not the main regression tables.  
Instead, they help interpret **which app characteristics tend to co-occur with higher privacy risk**.


In [ ]:

biz_df = app_level.copy()

summary_frames = []
for flag_col, title in [
    ("ad_supported_bin", "Ads"),
    ("offersIAP_bin", "In-app purchases"),
]:
    if flag_col in biz_df.columns:
        metric_cols = [c for c in ["ADII", "DGI", "PCLR", "AS"] if c in biz_df.columns]
        tmp = (
            biz_df.groupby(flag_col)[metric_cols]
            .median()
            .reset_index()
        )
        tmp["grouping"] = title
        summary_frames.append(tmp)

if summary_frames:
    monetization_summary = pd.concat(summary_frames, ignore_index=True)
    display(monetization_summary)
else:
    monetization_summary = pd.DataFrame()
    print("No monetization flags available for summary tables.")


In [ ]:

corr_cols = [c for c in [
    "ad_supported_bin", "offersIAP_bin",
    "num_trackers", "num_permissions", "num_dangerous_permissions",
    "log_downloads_for_model", "average_score",
    "ADII", "DGI", "PCLR", "AS"
] if c in biz_df.columns]

corr_df = biz_df[corr_cols].copy().dropna(how="all")

label_map = {
    "ad_supported_bin": "Has Ads",
    "offersIAP_bin": "Has IAP",
    "num_trackers": "Num Trackers",
    "num_permissions": "Num Permissions",
    "num_dangerous_permissions": "Num Dangerous Permissions",
    "log_downloads_for_model": "Log Downloads",
    "average_score": "Avg Score",
    "ADII": "ADII",
    "DGI": "DGI",
    "PCLR": "PCLR",
    "AS": "AS"
}

if len(corr_cols) >= 2:
    corr = corr_df.corr(numeric_only=True)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.grid(False)

    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="equal")

    labels = [label_map.get(c, c) for c in corr.columns]
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=12)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=12)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title("Correlation Heatmap: Ecosystem and Privacy Variables", fontsize=12, pad=12)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Correlation", rotation=270, labelpad=12)

    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            val = corr.iloc[i, j]
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center",
                fontsize=9,
                color="white" if abs(val) > 0.5 else "black"
            )

    plt.tight_layout()
    plt.savefig(FIG_DIR / "privacy_businessmodel_correlation_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()



## 12. Regression-ready narrative helper

This final cell creates a compact machine-readable summary of direction and significance for the main OLS results.  
It is useful when drafting Section 7.6 prose.


In [ ]:

def direction_label(x):
    if x > 0:
        return "positive"
    if x < 0:
        return "negative"
    return "null"

narrative_rows = []
for outcome in ["log_ADII", "DGI", "PCLR", "AS"]:
    sub = ols_results[
        (ols_results["outcome"] == outcome) &
        (ols_results["term"].isin(focus_terms))
    ].copy()

    for _, row in sub.iterrows():
        narrative_rows.append({
            "outcome": outcome,
            "term": row["term"],
            "direction": direction_label(row["coef"]),
            "significant_0_05": bool(row["p_value"] < 0.05),
            "coef": round(row["coef"], 4),
            "p_value": round(row["p_value"], 6),
        })

narrative_df = pd.DataFrame(narrative_rows)
display(narrative_df)

print("Figures saved to:", FIG_DIR.resolve())
